In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import chardet

In [2]:
# =========================================================
# CONFIGURACIÓN
# =========================================================
data_csv_files = [
    "../data/raw/Incidencia delictiva 2011-2025.csv",
    "../data/raw/SESNSP_incidencia_municipal.csv"
]

geojson_file = "../data/raw/mun_jal.geojson"

In [3]:
# =========================================================
# UTILIDADES DE CARGA
# =========================================================
def read_csv_chardet(file_path, sample_size=40000, **kwargs):
    """
    Lee un CSV detectando su encoding a partir de una muestra de bytes.
    """
    file_path = Path(file_path)

    with open(file_path, "rb") as f:
        raw_sample = f.read(sample_size)

    detected = chardet.detect(raw_sample)
    encoding = detected["encoding"]
    confidence = detected["confidence"]

    print(f"Archivo: {file_path.name}")
    print(f"Encoding detectado: {encoding} | confianza: {confidence:.4f}")
    print("-" * 60)

    df = pd.read_csv(file_path, encoding=encoding, **kwargs)
    return df


def load_datasets(file_paths):
    """
    Carga varios CSV en un diccionario:
    {nombre_archivo_sin_extension: dataframe}
    """
    dfs = {}
    for file_path in file_paths:
        file_name = Path(file_path).stem
        dfs[file_name] = read_csv_chardet(file_path)
    return dfs


In [4]:
dfs = load_datasets(data_csv_files)

print("Archivos cargados:")
print(list(dfs.keys()))

gdf = gpd.read_file(geojson_file)
print("GeoJSON leído correctamente")
display(gdf.head())

Archivo: Incidencia delictiva 2011-2025.csv
Encoding detectado: utf-16le | confianza: 0.8500
------------------------------------------------------------
Archivo: SESNSP_incidencia_municipal.csv
Encoding detectado: ISO-8859-1 | confianza: 0.7300
------------------------------------------------------------
Archivos cargados:
['Incidencia delictiva 2011-2025', 'SESNSP_incidencia_municipal']
GeoJSON leído correctamente


,cvegeo,cve_ent,cve_mun,nomgeo,cve_cab,pob_total,pob_femenina,pob_masculina,total_viviendas_habitadas,geometry
0,14089,14,089,Techaluta de Montenegro,0001,4072,2061,2011,1096,"MULTIPOLYGON (((-103.56717 20.14875, -103.5668..."
1,14094,14,094,Tequila,0001,44353,22462,21891,11150,"MULTIPOLYGON (((-103.53858 21.44441, -103.5384..."
2,14095,14,095,Teuchitlán,0001,9647,4938,4709,2712,"MULTIPOLYGON (((-103.78641 20.78581, -103.7868..."
3,14096,14,096,Tizapán el Alto,0001,22758,11579,11179,6317,"MULTIPOLYGON (((-103.03713 20.18173, -103.0365..."
4,14032,14,032,Chiquilistlán,0001,5983,3008,2975,1468,"MULTIPOLYGON (((-103.84083 20.20122, -103.8399..."


In [5]:
# =========================================================
# INSPECCIÓN GENERAL
# =========================================================
def resumen_dataframe(df, name="DataFrame", n=5):
    """
    Muestra resumen general de un DataFrame.
    """
    print("=" * 80)
    print(f"DATAFRAME: {name}")
    print("=" * 80)
    print(f"Dimensiones: {df.shape}")
    print("\nPrimeras filas:")
    display(df.head(n))
    print("\nColumnas:")
    print(df.columns.tolist())
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\n")


def clasificar_variables(df):
    """
    Clasifica columnas por tipo.
    Compatible con cambios recientes de pandas.
    """
    numericas = df.select_dtypes(include=["number"]).columns.tolist()
    categoricas = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    booleanas = df.select_dtypes(include=["bool"]).columns.tolist()

    posibles_fechas = [
        col for col in df.columns
        if any(palabra in str(col).lower() for palabra in ["fecha", "date", "año", "mes"])
    ]

    return {
        "numéricas": numericas,
        "categóricas": categoricas,
        "booleanas": booleanas,
        "posibles_fechas": posibles_fechas
    }


def imprimir_clasificacion(df, name="DataFrame"):
    """
    Imprime clasificación de variables.
    """
    clasif = clasificar_variables(df)

    print("=" * 80)
    print(f"CLASIFICACIÓN DE VARIABLES: {name}")
    print("=" * 80)

    for tipo, cols in clasif.items():
        print(f"\n{tipo.upper()} ({len(cols)}):")
        print(cols)


def resumen_missing(df, name="DataFrame"):
    """
    Muestra valores faltantes por columna.
    """
    print("=" * 80)
    print(f"DATOS PERDIDOS: {name}")
    print("=" * 80)

    missing_abs = df.isna().sum()
    missing_pct = (missing_abs / len(df)) * 100

    resumen = pd.DataFrame({
        "faltantes": missing_abs,
        "porcentaje_faltantes": missing_pct
    }).sort_values("porcentaje_faltantes", ascending=False)

    display(resumen[resumen["faltantes"] > 0])


def resumen_duplicados(df, name="DataFrame"):
    """
    Muestra cantidad de duplicados exactos.
    """
    print("=" * 80)
    print(f"DUPLICADOS: {name}")
    print("=" * 80)

    duplicados = df.duplicated().sum()
    print(f"Registros duplicados exactos: {duplicados}")

    if duplicados > 0:
        print("\nEjemplos de duplicados:")
        display(df[df.duplicated()].head())


def inspeccion_completa(dfs):
    """
    Ejecuta toda la inspección general para un diccionario de DataFrames.
    """
    for name, df in dfs.items():
        resumen_dataframe(df, name)
        imprimir_clasificacion(df, name)
        resumen_missing(df, name)
        resumen_duplicados(df, name)

In [6]:
inspeccion_completa(dfs)

DATAFRAME: Incidencia delictiva 2011-2025
Dimensiones: (19, 18)

Primeras filas:


,Indicador,Área geográfica,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 17
0,Gobierno Seguridad y Justicia > Seguridad púb...,14 Jalisco,,36.17941,,38.6693693345,,37.148155,,32.043963,,30.507775,,15.704449,,,,NaN
1,Gobierno Seguridad y Justicia > Seguridad púb...,14 Jalisco,,90.3157500671,,88.0911803147,,90.123581,,85.105378,,91.202488,,75.788450,,92.2475207453,,NaN
2,Gobierno Seguridad y Justicia > Seguridad púb...,14 Jalisco,,40.0917317739,,49.438378792,,28.223370,,64.963545,,53.421066,,59.471157,,,,NaN
3,Gobierno Seguridad y Justicia > Seguridad púb...,14 Jalisco,27.9689317086,30.1149808916,21.8701841561,37.2020343419,33.7281265338,31.318630,25.8448211729,31.136236,32.9858796488,34.795826,35.6069878472,30.730733,34.9958373939,30.3091452521,26.561150999,NaN
4,Gobierno Seguridad y Justicia > Seguridad púb...,14 Jalisco,93.7719006573,92.1601174791,93.7628569431,93.316267421,94.7909763847,94.243847,93.154160431,92.706276,91.7505284894,91.244284,92.8875652988,92.964244,89.5148129118,91.6092830274,91.7118242385,NaN



Columnas:
['Indicador', 'Área geográfica', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', 'Unnamed: 17']

Tipos de datos:
Indicador              str
Área geográfica        str
2010                   str
2011                   str
2012                   str
2013                   str
2014                   str
2015               float64
2016                   str
2017               float64
2018                   str
2019               float64
2020                   str
2021               float64
2022                   str
2023                   str
2024                   str
Unnamed: 17        float64
dtype: object


CLASIFICACIÓN DE VARIABLES: Incidencia delictiva 2011-2025

NUMÉRICAS (5):
['2015', '2017', '2019', '2021', 'Unnamed: 17']

CATEGÓRICAS (13):
['Indicador', 'Área geográfica', '2010', '2011', '2012', '2013', '2014', '2016', '2018', '2020', '2022', '2023', '2024']

BOOLEANAS (0):
[]

POSIBLES_FECHAS (0)

,faltantes,porcentaje_faltantes
Unnamed: 17,19,100.000000
2016,5,26.315789
2023,5,26.315789
2022,5,26.315789
2015,5,26.315789
2017,5,26.315789
2019,5,26.315789
2020,5,26.315789
2018,5,26.315789
2021,5,26.315789


DUPLICADOS: Incidencia delictiva 2011-2025
Registros duplicados exactos: 0
DATAFRAME: SESNSP_incidencia_municipal
Dimensiones: (2562994, 21)

Primeras filas:


,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,2,...,1,1,0,1,1,0,2,1,0,1
1,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,...,0,0,0,1,0,1,0,0,0,0
2,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,1,1,3,2,0,1,2,0,0,0
3,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,1,...,0,1,0,0,0,0,0,0,0,0
4,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,0,1,0,0,0,0,0,0,0



Columnas:
['Año', 'Clave_Ent', 'Entidad', 'Cve. Municipio', 'Municipio', 'Bien jurídico afectado', 'Tipo de delito', 'Subtipo de delito', 'Modalidad', 'Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

Tipos de datos:
Año                       int64
Clave_Ent                 int64
Entidad                     str
Cve. Municipio            int64
Municipio                   str
Bien jurídico afectado      str
Tipo de delito              str
Subtipo de delito           str
Modalidad                   str
Enero                     int64
Febrero                   int64
Marzo                     int64
Abril                     int64
Mayo                      int64
Junio                     int64
Julio                     int64
Agosto                    int64
Septiembre                int64
Octubre                   int64
Noviembre                 int64
Diciembre                 int64
dtype: object


CLASIFICACIÓN DE V

,faltantes,porcentaje_faltantes


DUPLICADOS: SESNSP_incidencia_municipal
Registros duplicados exactos: 0


In [7]:
def clean_incidencia_delictiva(df):
    """
    Limpia el dataset 'Incidencia delictiva 2011-2025'.

    - elimina columnas Unnamed
    - elimina filas de notas/fuentes
    - limpia la columna Indicador (solo último nivel)
    - convierte columnas de años a numérico
    - resetea índice
    """
    df = df.copy()

    # 1) Eliminar columnas basura tipo 'Unnamed'
    unnamed_cols = [col for col in df.columns if str(col).startswith("Unnamed")]
    df = df.drop(columns=unnamed_cols, errors="ignore")

    # 2) Conservar solo filas con área geográfica válida
    df = df[df["Área geográfica"].notna()].copy()

    # 3) Quitar notas, fuentes y líneas auxiliares
    patrones_basura = r"^(?:Notas|Fuente|/)"
    df = df[
        ~df["Indicador"].astype(str).str.strip().str.contains(
            patrones_basura, case=False, na=False, regex=True
        )
    ].copy()

    # 4) Limpiar el nombre del indicador (último nivel)
    df["Indicador"] = (
        df["Indicador"]
        .astype(str)
        .str.split(">")
        .str[-1]
        .str.strip()
    )

    # 5) Convertir columnas de años a numérico
    cols_anios = [col for col in df.columns if str(col).isdigit()]
    df[cols_anios] = df[cols_anios].apply(pd.to_numeric, errors="coerce")

    # 6) Resetear índice
    df = df.reset_index(drop=True)

    return df

In [8]:
df_indicadores = clean_incidencia_delictiva(dfs["Incidencia delictiva 2011-2025"])

In [9]:
print("Shape original:", dfs["Incidencia delictiva 2011-2025"].shape)
print("Shape limpio:", df_indicadores.shape)

display(df_indicadores.head(5))
display(df_indicadores.tail(5))
print(df_indicadores.dtypes)

Shape original: (19, 18)
Shape limpio: (14, 17)


,Indicador,Área geográfica,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Porcentaje de unidades económicas víctimas del...,14 Jalisco,NaN,36.179410,NaN,38.669369,NaN,37.148155,NaN,32.043963,NaN,30.507775,NaN,15.704449,NaN,NaN,NaN
1,Cifra negra de delitos en unidades económicas ...,14 Jalisco,NaN,90.315750,NaN,88.091180,NaN,90.123581,NaN,85.105378,NaN,91.202488,NaN,75.788450,NaN,92.247521,NaN
2,Porcentaje de delitos con portación de armas e...,14 Jalisco,NaN,40.091732,NaN,49.438379,NaN,28.223370,NaN,64.963545,NaN,53.421066,NaN,59.471157,NaN,NaN,NaN
3,Porcentaje de delitos con portación de armas e...,14 Jalisco,27.968932,30.114981,21.870184,37.202034,33.728127,31.318630,25.844821,31.136236,32.985880,34.795826,35.606988,30.730733,34.995837,30.309145,26.561151
4,Cifra Negra (delitos no denunciados y los deli...,14 Jalisco,93.771901,92.160117,93.762857,93.316267,94.790976,94.243847,93.154160,92.706276,91.750528,91.244284,92.887565,92.964244,89.514813,91.609283,91.711824


,Indicador,Área geográfica,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
9,Tasa de incidencia delictiva por cada cien mil...,14 Jalisco,32980.499401,29351.269584,49083.090096,47278.068799,43076.296792,49316.850173,41874.3805,43022.896851,40542.747955,34703.076914,33247.921375,31944.188360,28925.655326,31731.225728,33328.978807
10,Tasa de concentración de delitos ocurridos (Ta...,14 Jalisco,NaN,2.776640,NaN,1.574300,NaN,1.722755,NaN,2.080048,NaN,1.810368,NaN,2.190885,NaN,NaN,NaN
11,Delitos denunciados ante el Ministerio Público...,14 Jalisco,NaN,6240.433540,NaN,6251.144610,NaN,3610.065499,NaN,5349.957410,NaN,2379.910231,NaN,2433.822100,NaN,NaN,NaN
12,Delitos no denunciados o en los que no se espe...,14 Jalisco,NaN,239326.636240,NaN,144235.866150,NaN,194223.514887,NaN,184122.186274,NaN,175090.162106,NaN,85500.886900,NaN,NaN,NaN
13,Delitos ocurridos a unidades económicas (Delit...,14 Jalisco,NaN,271898.389370,NaN,170830.961990,NaN,219513.669461,NaN,222632.396873,NaN,194589.069317,NaN,116026.530500,NaN,212744.197800,NaN


Indicador           object
Área geográfica        str
2010               float64
2011               float64
2012               float64
2013               float64
2014               float64
2015               float64
2016               float64
2017               float64
2018               float64
2019               float64
2020               float64
2021               float64
2022               float64
2023               float64
2024               float64
dtype: object


In [10]:
# 1. melt
df_long = df_indicadores.melt(
    id_vars=["Indicador", "Área geográfica"],
    var_name="Año",
    value_name="Valor"
)

# 2. eliminar NaN reales
df_long = df_long.dropna(subset=["Valor"])

In [11]:
df_long

,Indicador,Área geográfica,Año,Valor
3,Porcentaje de delitos con portación de armas e...,14 Jalisco,2010,2.796893e+01
4,Cifra Negra (delitos no denunciados y los deli...,14 Jalisco,2010,9.377190e+01
5,Porcentaje de hogares víctimas del delito (Por...,14 Jalisco,2010,3.904063e+01
6,Delitos denunciados (Delitos) Anual /f1,14 Jalisco,2010,1.631110e+05
7,Delitos no denunciados o en los que no se espe...,14 Jalisco,2010,1.478963e+06
...,...,...,...,...
201,Porcentaje de hogares víctimas del delito (Por...,14 Jalisco,2024,2.971865e+01
202,Delitos denunciados (Delitos) Anual /f1,14 Jalisco,2024,2.089800e+05
203,Delitos no denunciados o en los que no se espe...,14 Jalisco,2024,1.936655e+06
204,Delitos denunciados ante el Ministerio Público...,14 Jalisco,2024,3.114600e+04


In [12]:
df_long["Indicador"] = (
    df_long["Indicador"]
    .str.replace(r"/f\d+", "", regex=True)
    .str.replace(r"/[a-z]\d*", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [13]:
print(sorted(df_long["Indicador"].unique()))

['Cifra Negra (delitos no denunciados y los delitos denunciados que no tuvieron inicio de averiguación previa) (Porcentaje) Anual', 'Cifra negra de delitos en unidades económicas (Porcentaje) Bienal', 'Delitos denunciados (Delitos) Anual', 'Delitos denunciados ante el Ministerio Público a partir de los cuales no se inició averiguación previa o no se especifica si se inició averiguación previa (Delitos) Anual', 'Delitos denunciados ante el Ministerio Público a partir de los cuales no se inició averiguación previa o no se especifica si se inició averiguación previa (en unidades económicas) (Delitos) Bienal', 'Delitos no denunciados o en los que no se especifica si se denunció ante el Ministerio Público (Delitos) Anual', 'Delitos no denunciados o en los que no se especifica si se denunció ante el Ministerio Público (en unidades económicas) (Delitos) Bienal', 'Delitos ocurridos a unidades económicas (Delitos) Bienal', 'Porcentaje de delitos con portación de armas en personas de 18 años y m

In [14]:
#Existen datos Bienales (2 años) entonces por eso tanto NaN. Vamos a separar anuales y bienales
df_anual = df_long[df_long["Indicador"].str.contains("Anual")].copy()
df_bienal = df_long[df_long["Indicador"].str.contains("Bienal")].copy()
df_anual["Año"] = df_anual["Año"].astype(int)

print("ANUAL:", df_anual.shape)
print("BIENAL:", df_bienal.shape)

print("\nIndicadores ANUAL:")
print(df_anual["Indicador"].nunique())

print("\nIndicadores BIENAL:")
print(df_bienal["Indicador"].nunique())

ANUAL: (105, 4)
BIENAL: (44, 4)

Indicadores ANUAL:
7

Indicadores BIENAL:
7


In [15]:
df_anual.groupby("Indicador")["Año"].nunique()


Indicador
Cifra Negra (delitos no denunciados y los delitos denunciados que no tuvieron inicio de averiguación previa) (Porcentaje) Anual                                               15
Delitos denunciados (Delitos) Anual                                                                                                                                           15
Delitos denunciados ante el Ministerio Público a partir de los cuales no se inició averiguación previa o no se especifica si se inició averiguación previa (Delitos) Anual    15
Delitos no denunciados o en los que no se especifica si se denunció ante el Ministerio Público (Delitos) Anual                                                                15
Porcentaje de delitos con portación de armas en personas de 18 años y más (Porcentaje) Anual                                                                                  15
Porcentaje de hogares víctimas del delito (Porcentaje) Anual                                             

In [16]:
#Vamos con el otro dataset
df_delitos = dfs["SESNSP_incidencia_municipal"].copy()

df_delitos = df_delitos[df_delitos["Entidad"] == "Jalisco"]
df_delitos

,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
52136,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,0,1
52137,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,0,...,0,0,0,0,0,0,0,0,0,0
52138,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,0,0,0,0,0,0,1,0,0,0
52139,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,0,...,0,0,0,0,0,0,0,0,0,0
52140,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2385217,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Falsificación,Falsificación,Falsificación,0,...,0,0,0,0,0,0,0,0,0,0
2385218,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Contra el medio ambiente,Contra el medio ambiente,Contra el medio ambiente,0,...,0,0,0,0,0,0,0,0,0,0
2385219,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,0,...,0,0,0,0,0,0,0,0,0,0
2385220,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
df_delitos.info()

<class 'pandas.DataFrame'>
Index: 135828 entries, 52136 to 2385221
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Año                     135828 non-null  int64
 1   Clave_Ent               135828 non-null  int64
 2   Entidad                 135828 non-null  str  
 3   Cve. Municipio          135828 non-null  int64
 4   Municipio               135828 non-null  str  
 5   Bien jurídico afectado  135828 non-null  str  
 6   Tipo de delito          135828 non-null  str  
 7   Subtipo de delito       135828 non-null  str  
 8   Modalidad               135828 non-null  str  
 9   Enero                   135828 non-null  int64
 10  Febrero                 135828 non-null  int64
 11  Marzo                   135828 non-null  int64
 12  Abril                   135828 non-null  int64
 13  Mayo                    135828 non-null  int64
 14  Junio                   135828 non-null  int64
 15  Julio      

In [18]:
cols_meses = [
    "Enero","Febrero","Marzo","Abril","Mayo","Junio",
    "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"
]

df_delitos["Total_Anual"] = df_delitos[cols_meses].sum(axis=1)

In [19]:
df_delitos

,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Total_Anual
52136,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,1,1
52137,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,0,...,0,0,0,0,0,0,0,0,0,0
52138,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,0,0,0,0,0,1,0,0,0,1
52139,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,0,...,0,0,0,0,0,0,0,0,0,0
52140,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2385217,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Falsificación,Falsificación,Falsificación,0,...,0,0,0,0,0,0,0,0,0,0
2385218,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Contra el medio ambiente,Contra el medio ambiente,Contra el medio ambiente,0,...,0,0,0,0,0,0,0,0,0,0
2385219,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,0,...,0,0,0,0,0,0,0,0,0,0
2385220,2025,14,Jalisco,14998,No Especificado,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
#Hay municipios con clave pero nombre no especificado, veamos si lo podemos reemplazar por el nombre que corresponde basados del geojson
print(gdf.columns.tolist())
display(gdf.head())

['cvegeo', 'cve_ent', 'cve_mun', 'nomgeo', 'cve_cab', 'pob_total', 'pob_femenina', 'pob_masculina', 'total_viviendas_habitadas', 'geometry']


,cvegeo,cve_ent,cve_mun,nomgeo,cve_cab,pob_total,pob_femenina,pob_masculina,total_viviendas_habitadas,geometry
0,14089,14,089,Techaluta de Montenegro,0001,4072,2061,2011,1096,"MULTIPOLYGON (((-103.56717 20.14875, -103.5668..."
1,14094,14,094,Tequila,0001,44353,22462,21891,11150,"MULTIPOLYGON (((-103.53858 21.44441, -103.5384..."
2,14095,14,095,Teuchitlán,0001,9647,4938,4709,2712,"MULTIPOLYGON (((-103.78641 20.78581, -103.7868..."
3,14096,14,096,Tizapán el Alto,0001,22758,11579,11179,6317,"MULTIPOLYGON (((-103.03713 20.18173, -103.0365..."
4,14032,14,032,Chiquilistlán,0001,5983,3008,2975,1468,"MULTIPOLYGON (((-103.84083 20.20122, -103.8399..."


In [21]:
# =========================================================
# Corregir municipios faltantes / "No especificado" usando el geojson
# =========================================================

# 1. Crear catálogo clave -> nombre desde el geojson
catalogo_geo = (
    gdf[["cvegeo", "nomgeo"]]
    .drop_duplicates()
    .rename(columns={
        "cvegeo": "Cve. Municipio",
        "nomgeo": "Municipio_geo"
    })
)

# 2. Homologar tipos de clave
df_delitos["Cve. Municipio"] = df_delitos["Cve. Municipio"].astype(str).str.strip()
catalogo_geo["Cve. Municipio"] = catalogo_geo["Cve. Municipio"].astype(str).str.strip()

# 3. Merge con catálogo geográfico
df_delitos = df_delitos.merge(
    catalogo_geo,
    on="Cve. Municipio",
    how="left"
)

# 4. Detectar registros donde Municipio está mal:
#    - NaN
#    - "No especificado"
mask_municipio_invalido = (
    df_delitos["Municipio"].isna() |
    df_delitos["Municipio"].astype(str).str.strip().str.lower().eq("no especificado")
)

# 5. Reemplazar SOLO si existe nombre válido en el geojson
df_delitos["Municipio"] = df_delitos["Municipio"].mask(
    mask_municipio_invalido & df_delitos["Municipio_geo"].notna(),
    df_delitos["Municipio_geo"]
)

# 6. Separar registros con y sin match geográfico
df_delitos_validos = df_delitos[df_delitos["Municipio_geo"].notna()].copy()
df_delitos_sin_match = df_delitos[df_delitos["Municipio_geo"].isna()].copy()

# 7. Quitar columna auxiliar en ambos
df_delitos = df_delitos.drop(columns=["Municipio_geo"])
df_delitos_validos = df_delitos_validos.drop(columns=["Municipio_geo"])
df_delitos_sin_match = df_delitos_sin_match.drop(columns=["Municipio_geo"])

# 8. Verificaciones
print("Shape total df_delitos:", df_delitos.shape)
print("Shape válidos:", df_delitos_validos.shape)
print("Shape sin match:", df_delitos_sin_match.shape)

print(
    "\nMunicipios aún inválidos en df_delitos:",
    (
        df_delitos["Municipio"].isna() |
        df_delitos["Municipio"].astype(str).str.strip().str.lower().eq("no especificado")
    ).sum()
)

print("\nClaves sin match geográfico:")
print(sorted(df_delitos_sin_match["Cve. Municipio"].unique())[:20])

print("\nEjemplo de claves corregidas:")
display(
    df_delitos_validos[["Cve. Municipio", "Municipio"]]
    .drop_duplicates()
    .sort_values("Cve. Municipio")
    .head(20)
)

print("\nEjemplo de registros sin match:")
display(
    df_delitos_sin_match[["Cve. Municipio", "Municipio"]]
    .drop_duplicates()
    .sort_values("Cve. Municipio")
    .head(20)
)

Shape total df_delitos: (135828, 22)
Shape válidos: (134750, 22)
Shape sin match: (1078, 22)

Municipios aún inválidos en df_delitos: 1078

Claves sin match geográfico:
['14998']

Ejemplo de claves corregidas:


,Cve. Municipio,Municipio
0,14001,Acatic
98,14002,Acatlán de Juárez
196,14003,Ahualulco de Mercado
294,14004,Amacueca
392,14005,Amatitán
490,14006,Ameca
588,14007,San Juanito de Escobedo
686,14008,Arandas
784,14009,El Arenal
882,14010,Atemajac de Brizuela



Ejemplo de registros sin match:


,Cve. Municipio,Municipio
12250,14998,No Especificado


In [22]:
df_delitos_validos

,Año,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Total_Anual
0,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,1,1
1,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,0,...,0,0,0,0,0,0,0,0,0,0
2,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,0,0,0,0,0,1,0,0,0,1
3,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,0,...,0,0,0,0,0,0,0,0,0,0
4,2015,14,Jalisco,14001,Acatic,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135725,2025,14,Jalisco,14125,San Ignacio Cerro Gordo,Otros bienes jurídicos afectados (del fuero co...,Falsificación,Falsificación,Falsificación,0,...,0,0,0,0,0,0,1,0,0,1
135726,2025,14,Jalisco,14125,San Ignacio Cerro Gordo,Otros bienes jurídicos afectados (del fuero co...,Contra el medio ambiente,Contra el medio ambiente,Contra el medio ambiente,0,...,0,0,0,0,0,0,0,0,0,0
135727,2025,14,Jalisco,14125,San Ignacio Cerro Gordo,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,0,...,0,0,0,0,0,0,0,0,0,0
135728,2025,14,Jalisco,14125,San Ignacio Cerro Gordo,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
from pathlib import Path

processed_dir = Path("../data/interim")
processed_dir.mkdir(parents=True, exist_ok=True)

df_anual.to_csv(processed_dir / "indicadores_anuales_jalisco.csv", index=False, encoding="utf-8-sig")
df_bienal.to_csv(processed_dir / "indicadores_bienales_jalisco.csv", index=False, encoding="utf-8-sig")
df_delitos_validos.to_csv(processed_dir / "delitos_jalisco_validos.csv", index=False, encoding="utf-8-sig")
df_delitos_sin_match.to_csv(processed_dir / "delitos_jalisco_sin_match_geo.csv", index=False, encoding="utf-8-sig")